# Tema: Permisos, máscaras y gobierno

## Objetivos
Diseñar mínimo privilegio, aplicar políticas por fila/columna y distinguir REVOKE de DENY.

## Conceptos importantes para el examen
Privilegios heredados; grupos e identidades de servicio; propietarios; row filters; column masks; ABAC por tags; auditoría y linaje.

**Dificultad:** Examen · **Tiempo estimado:** 90 min.

Las UDF requieren CREATE FUNCTION y EXECUTE; adjuntar políticas requiere propiedad o privilegios apropiados y cómputo compatible. GRANT/REVOKE necesitan autoridad sobre el objeto y grupos existentes. Las ampliaciones ABAC/DENY se dejan optativas por disponibilidad y permisos.

Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_25_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
employees = spark.createDataFrame(
    [(i, f"Empleado {i:02d}", ["Data", "Sales", "Finance"][i % 3],
      30000 + i * 1500, i % 4 != 0,
      datetime(2026, 1, 1), datetime(2026, 1, 1)) for i in range(1, 19)],
    "employee_id INT, name STRING, department STRING, salary INT, active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
)
employees.createOrReplaceTempView("employees_seed")
employees.write.format("delta").mode("errorifexists").saveAsTable("employees")
display(employees.orderBy("employee_id"))

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Revisar identidad y permisos

In [ ]:
display(spark.sql("SELECT current_user()"))
display(spark.sql("SHOW GRANTS ON TABLE employees"))

### 2. Vista de consumo sin salarios
Dar acceso a la vista y no a la tabla puede limitar columnas. El propietario conserva acceso a su tabla.

In [ ]:
%sql
CREATE OR REPLACE VIEW employees_public AS SELECT employee_id, name, department FROM employees;
SELECT * FROM employees_public;

### 3. Simulación de una máscara
Evalúa dos perfiles ficticios. Esta simulación no cambia permisos de UC.

In [ ]:
display(employees.select("employee_id", F.lit(None).cast("int").alias("salary_for_reader"), F.col("salary").alias("salary_for_hr")))

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Genera y, con autoridad, ejecuta permisos mínimos de lector para employees_public y de productor para employees.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Revoca el SELECT directo del lector sobre employees si existiera y revisa herencia. Explica por qué podría seguir leyendo.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
En una copia de employees, crea UDF de máscara salarial para el grupo dea_hr y filtro que permita Data a otros lectores. Adjunta ambas si tienes permisos.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Diseña una política ABAC para columnas sensibles usando un tag gobernado. En Catalog Explorer configura la política y conserva el SQL generado.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Compara REVOKE con DENY y prepara un ejemplo ABAC DENY de administración de acceso solo en el schema de laboratorio.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 6
Consulta linaje de employees_public y prepara una consulta de auditoría limitada. Distingue compartir de federar.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** USE en contenedores; SELECT en vista; SELECT/MODIFY en tabla para productor.

**Pista 2:** REVOKE no anula permisos que llegan por otro grupo o contenedor.

**Pista 3:** is_account_group_member determina pertenencia; sin permisos usa la simulación local.

**Pista 4:** ABAC centraliza selección por tags; una máscara manual solo está adjunta a su tabla.

**Pista 5:** La disponibilidad de DENY en UC es específica: no inventes DENY SELECT genérico.

**Pista 6:** Linaje en Catalog Explorer; system.access.audit requiere privilegios.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
READER, WRITER = "dea_readers", "dea_writers"
APPLY_GRANTS = False
commands = []
for group in [READER,WRITER]:
    commands += [f"GRANT USE CATALOG ON CATALOG {ident(CATALOG)} TO {ident(group)}", f"GRANT USE SCHEMA ON SCHEMA {ident(CATALOG)}.{ident(SCHEMA)} TO {ident(group)}"]
commands += [f"GRANT SELECT ON VIEW employees_public TO {ident(READER)}", f"GRANT SELECT, MODIFY ON TABLE employees TO {ident(WRITER)}"]
for command in commands:
    print(command)
    if APPLY_GRANTS:
        spark.sql(command)

### Solución 2

In [ ]:
command = f"REVOKE SELECT ON TABLE employees FROM {ident(READER)}"
print(command)
if APPLY_GRANTS:
    spark.sql(command)
display(spark.sql("SHOW GRANTS ON TABLE employees"))
display(spark.sql(f"SHOW GRANTS ON SCHEMA {ident(CATALOG)}.{ident(SCHEMA)}"))
# Probar con otra identidad autorizada; no simules una identidad cambiando current_user.

### Solución 3

In [ ]:
APPLY_POLICIES = False
policy_sql = [
 "CREATE OR REPLACE TABLE protected_employees USING DELTA AS SELECT * FROM employees",
 """CREATE OR REPLACE FUNCTION salary_mask(value INT) RETURNS INT
 RETURN CASE WHEN is_account_group_member('dea_hr') THEN value ELSE CAST(NULL AS INT) END""",
 """CREATE OR REPLACE FUNCTION department_filter(value STRING) RETURNS BOOLEAN
 RETURN is_account_group_member('dea_hr') OR value = 'Data'""",
 "ALTER TABLE protected_employees ALTER COLUMN salary SET MASK salary_mask",
 "ALTER TABLE protected_employees SET ROW FILTER department_filter ON (department)"]
for command in policy_sql:
    print(command)
    if APPLY_POLICIES:
        spark.sql(command)
if APPLY_POLICIES:
    display(spark.table("protected_employees"))
else:
    display(employees.filter("department='Data'").withColumn("salary",F.lit(None).cast("int")))

### Solución 4

In [ ]:
# Ruta real (requiere permisos y función/tag existentes):
# Catalog Explorer → schema de prácticas → Policies → New policy → Column mask.
# Principal: grupo lector; función salary_mask; columnas INT con tag gobernado de sensibilidad.
# Etiqueta salary en protected_employees, revisa Show code y aplica si tu rol lo permite.
# Verifica con dos identidades autorizadas; revisa políticas efectivas y herencia.
# Alternativa ejecutable sin permisos: seleccionar por metadatos ficticios y aplicar máscara.
tags = {"salary":"sensitive", "name":"public"}
masked = employees.select(*[F.lit(None).cast("int").alias(c) if tags.get(c)=="sensitive" else F.col(c) for c in employees.columns])
display(masked)

### Solución 5

In [ ]:
# Ampliación Beta: requiere soporte ABAC DENY y compute clásico DBR 18 LTS+.
# Alcance actual documentado: MANAGE ACCESS CONTROL, no SELECT/MODIFY.
APPLY_DENY = False
command = f"""CREATE POLICY restrict_access_admin ON SCHEMA {ident(CATALOG)}.{ident(SCHEMA)}
TO {ident(READER)} DENY MANAGE ACCESS CONTROL FOR TABLES"""
print(command)
if APPLY_DENY:
    spark.sql(command)
    display(spark.sql(f"SHOW EFFECTIVE POLICIES ON SCHEMA {ident(CATALOG)}.{ident(SCHEMA)}"))
    spark.sql(f"DROP POLICY restrict_access_admin ON SCHEMA {ident(CATALOG)}.{ident(SCHEMA)}")
# REVOKE quita una concesión; DENY soportado es una prohibición explícita.
# Nunca uses este ejercicio sobre un schema compartido de producción.

### Solución 6

In [ ]:
READ_SYSTEM_TABLES = False
query = """SELECT event_time, service_name, action_name FROM system.access.audit
WHERE event_date >= current_date() - INTERVAL 1 DAY ORDER BY event_time DESC LIMIT 20"""
print(query)
if READ_SYSTEM_TABLES:
    display(spark.sql(query))
# Catalog Explorer → employees_public → Lineage: localizar employees.
# Delta Sharing publica acceso compartido a datos; no concede escritura de negocio al receptor.
# Lakehouse Federation consulta fuentes remotas mediante conexiones y catálogos extranjeros.
# Conectar o compartir de verdad necesita infraestructura y privilegios externos.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
Tras REVOKE directo, un usuario conserva SELECT por un grupo. ¿Por qué?

A. REVOKE nunca funciona

B. Existe otra concesión efectiva

C. Delta ignora UC

D. Una vista siempre concede MODIFY

### Pregunta 2
¿Qué permite aplicar máscaras de forma central según etiquetas?

A. VACUUM

B. Un checkpoint

C. ABAC

D. INSERT OVERWRITE

### Pregunta 3
¿Cómo pruebas un filtro por grupos?

A. Con identidades autorizadas de perfiles diferentes

B. Cambiando una variable Python llamada user

C. Solo con DESCRIBE HISTORY

D. Con OPTIMIZE

### Respuestas y explicación
**1. B** — Los privilegios pueden provenir de múltiples rutas.

**2. C** — Las políticas usan atributos/tags de objetos.

**3. A** — La política se evalúa con la identidad real del consumidor.

### Documentación oficial
- [Máscaras y filtros](https://docs.databricks.com/aws/en/data-governance/unity-catalog/filters-and-masks/manually-apply)
- [DENY: alcance y requisitos](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/deny-policies)

## PARTE 6 - RETO FINAL
Publica una vista Gold para lectores y protege una tabla de detalle con política por columna o una alternativa de vista. Documenta permisos efectivos y verifica al menos dos perfiles si tienes identidades disponibles.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
